# Run 3 Results Analysis

Analysis of `outputs/run3_results_20260309_0709.csv`.

Run 3 used a 10-field output schema with numeric confidence (1-10), execution_path, merchant extraction, and a parallel naive LLM call for pipeline lift measurement.

**Sections:**
1. Setup and load
2. Taxonomy join and accuracy eval
3. Mismatch overview
4. Top mismatch pairs
5. Confusion matrix
6. Confidence vs accuracy
7. Execution path / pipeline coverage
8. Naive vs pipeline lift
9. Merchant extraction coverage
10. Provider breakdown
11. Spend type mismatch deep dive
12. Summary
13. Interactive review widget

## 1. Setup and Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 60)

RUN3_OUTPUT   = 'outputs/run3_results_20260309_1245.csv'
TAXONOMY_PATH = 'assets/taxonomy_master_updated_override.csv'

df = pd.read_csv(RUN3_OUTPUT)

# Confidence is numeric 1-10 in Run 3
df['confidence'] = pd.to_numeric(df['confidence'], errors='coerce')

print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
print(f'\nParse errors:  {df["parse_error"].sum()} ({df["parse_error"].mean()*100:.1f}%)')
print(f'Invalid keys:  {df["invalid_key"].sum()} ({df["invalid_key"].mean()*100:.1f}%)')
print(f'Confidence non-null: {df["confidence"].notna().sum()} / {len(df)}')
print(f'\nConfidence distribution:')
print(df['confidence'].value_counts().sort_index())

## 2. Taxonomy Join and Accuracy Eval

GT spend_type, category_type, and category_group are joined from taxonomy for L1/L2a/L3 evaluation.

Note: `gt_spend_type`, `gt_category_type`, and `base_category_key_gt` are already present in the Run 3 output - this cell confirms them and fills any gaps.

In [ ]:
tax = pd.read_csv(TAXONOMY_PATH)

# Build GT lookup from taxonomy (keyed on base_category_key)
gt_lookup = tax.set_index('base_category_key')[[
    'spend_type', 'leans', 'manual_spend_override', 'category_group', 'category_type'
]].rename(columns={
    'spend_type':            'tax_spend_type',
    'leans':                 'tax_leans',
    'manual_spend_override': 'tax_spend_override',
    'category_group':        'tax_category_group',
    'category_type':         'tax_category_type',
})
gt_lookup = gt_lookup[~gt_lookup.index.duplicated(keep='first')]

# Merge GT taxonomy fields onto GT key
df = df.merge(gt_lookup, left_on='gt_base_category_key', right_index=True, how='left')

# Resolve mixed GT spend_type via leans / manual_spend_override
def resolve_spend_type(row):
    existing = str(row.get('gt_spend_type', '')).lower().strip()
    if existing in ('spend', 'non_spend'):
        return existing
    raw = str(row.get('tax_spend_type', '')).lower().strip()
    if raw not in ('mixed', 'nan', ''):
        return raw
    override = str(row.get('tax_spend_override', '')).lower().strip()
    if override in ('spend', 'non_spend'):
        return override
    leans = str(row.get('tax_leans', '')).lower().strip()
    if leans in ('spend', 'non_spend'):
        return leans
    return raw

df['gt_spend_type_resolved'] = df.apply(resolve_spend_type, axis=1)

# Also get pred taxonomy fields (for L2 eval against predicted key)
pred_lookup = tax.set_index('base_category_key')[['category_group', 'category_type']].rename(columns={
    'category_group': 'pred_category_group',
    'category_type':  'pred_category_type',
})
pred_lookup = pred_lookup[~pred_lookup.index.duplicated(keep='first')]
df = df.merge(pred_lookup, left_on='base_category_key', right_index=True, how='left')

# --- L1: spend_type ---
l1_mask = df['gt_spend_type_resolved'].isin(['spend', 'non_spend'])
df['l1_match'] = df['spend_type'] == df['gt_spend_type_resolved']

# --- L2a: category_type (non-spend only) ---
gt_cat_type_col = 'gt_category_type' if 'gt_category_type' in df.columns else 'tax_category_type'
l2a_mask = df['gt_spend_type_resolved'] == 'non_spend'
df['l2a_match'] = (
    df['pred_category_type'].notna() &
    (df['pred_category_type'] == df[gt_cat_type_col])
)

# --- L3: exact key ---
l3_mask = ~df['parse_error'] & ~df['invalid_key']
df['l3_match'] = df['base_category_key'] == df['gt_base_category_key']

l1_acc  = df.loc[l1_mask, 'l1_match'].mean()
l2a_acc = df.loc[l2a_mask, 'l2a_match'].mean()
l3_acc  = df.loc[l3_mask, 'l3_match'].mean()

print('=== Accuracy Summary ===')
print(f'L1 spend_type  ({l1_mask.sum()} evaluable): {l1_acc:.1%}')
print(f'L2a cat_type   ({l2a_mask.sum()} evaluable, non-spend only): {l2a_acc:.1%}')
print(f'L3 exact key   ({l3_mask.sum()} evaluable): {l3_acc:.1%}')
print(f'\nUnresolvable GT spend_type (excluded from L1): {(~l1_mask).sum()}')
print(f'\nGT spend_type breakdown:')
print(df['gt_spend_type_resolved'].value_counts(dropna=False))


In [ ]:
print([c for c in df.columns if 'category_type' in c])

## 3. Mismatch Overview

In [ ]:
tax = pd.read_csv(TAXONOMY_PATH)
print(tax['base_category_key'].duplicated().sum())

In [ ]:
mismatches = df[~df['l3_match'] & l3_mask].copy()
matches    = df[ df['l3_match'] & l3_mask].copy()

l1_wrong = mismatches[~mismatches['l1_match'] & l1_mask.reindex(mismatches.index, fill_value=False)]
l1_right = mismatches[ mismatches['l1_match']]

print(f'Evaluable rows: {l3_mask.sum()}')
print(f'Matches:        {len(matches):,} ({len(matches)/l3_mask.sum():.1%})')
print(f'Mismatches:     {len(mismatches):,} ({len(mismatches)/l3_mask.sum():.1%})')
print(f'  Wrong spend_type:      {len(l1_wrong)} ({len(l1_wrong)/max(len(mismatches),1):.0%} of mismatches)')
print(f'  Right type, wrong key: {len(l1_right)} ({len(l1_right)/max(len(mismatches),1):.0%} of mismatches)')

cols_review = ['original_description', 'merchant', 'amount', 'gt_base_category_key',
               'base_category_key', 'reason', 'confidence', 'spend_type', 'gt_spend_type_resolved']
               
display(mismatches[cols_review].sort_values('gt_base_category_key').head(40))

## 4. Top Mismatch Pairs

In [ ]:
pair_counts = (
    mismatches
    .groupby(['gt_base_category_key', 'base_category_key'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
print('=== Top 20 Mismatch Pairs ===')
display(pair_counts.head(20))

## 5. Confusion Matrix (top keys)

In [ ]:
# Top 30 mismatch pairs
top_pairs = (
    mismatch_df.groupby(['gt_base_category_key', 'base_category_key'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(30)
)

top_gt   = top_pairs['gt_base_category_key'].unique()
top_pred = top_pairs['base_category_key'].unique()

cm_df = mismatch_df[
    mismatch_df['gt_base_category_key'].isin(top_gt) &
    mismatch_df['base_category_key'].isin(top_pred)
]
cm = pd.crosstab(cm_df['gt_base_category_key'], cm_df['base_category_key'])

fig, ax = plt.subplots(figsize=(18, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', linewidths=0.3, ax=ax,
            cbar_kws={'shrink': 0.5})
ax.set_title('Run 3 Confusion Matrix (top 30 mismatch pairs)', fontsize=13)
ax.set_xlabel('Predicted key')
ax.set_ylabel('Ground truth key')
plt.tight_layout()
plt.show()

## 6. Confidence vs Accuracy

Confidence is numeric 1-10 in Run 3. Analysed per integer value, not binned.

In [ ]:
df_conf = df[l3_mask].dropna(subset=['confidence']).copy()
df_conf['confidence'] = df_conf['confidence'].astype(int)

conf_summary = (
    df_conf.groupby('confidence')
    .agg(count=('l3_match', 'size'), accuracy=('l3_match', 'mean'))
    .reset_index()
)
conf_summary['accuracy_pct'] = (conf_summary['accuracy'] * 100).round(1)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.bar(conf_summary['confidence'], conf_summary['count'], alpha=0.35, color='steelblue', label='Row count')
ax2.plot(conf_summary['confidence'], conf_summary['accuracy'], 'o-', color='crimson', label='L3 accuracy')
ax1.set_xlabel('Confidence (1-10)')
ax1.set_ylabel('Row count', color='steelblue')
ax2.set_ylabel('L3 accuracy', color='crimson')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax1.set_title('Confidence Distribution vs L3 Accuracy - Run 3')
fig.tight_layout()
plt.show()

print('\nConfidence vs L3 accuracy (per integer value):')
display(conf_summary[['confidence', 'count', 'accuracy_pct']].rename(columns={'accuracy_pct': 'l3_accuracy_%'}))
print(f'\nOverall avg confidence: {df_conf["confidence"].mean():.2f}')

## 7. Execution Path / Pipeline Coverage

Coverage derived from `execution_path` field (e.g. `biller->llm`, `dt->regex->llm`, `llm`).

In [ ]:
ep = df['execution_path'].fillna('').str.lower()

biller_cov  = ep.str.startswith('biller')
mcc_cov     = ep.str.contains('mcc',      na=False)
dt_cov      = ep.str.contains('dt',       na=False)
merchant_cov= ep.str.contains('merchant', na=False)
regex_cov   = ep.str.contains('regex',    na=False)
llm_only    = ep.str.strip() == 'llm'

def pct(mask): return f'{mask.sum()} ({mask.mean()*100:.1f}%)'

print('Pipeline step coverage:')
print(f'  biller (short-circuit):  {pct(biller_cov)}')
print(f'  mcc hint:                {pct(mcc_cov)}')
print(f'  dt hint:                 {pct(dt_cov)}')
print(f'  merchant present:        {pct(merchant_cov)}')
print(f'  regex hint:              {pct(regex_cov)}')
print(f'  llm-only (no hints):     {pct(llm_only)}')

print('\nTop 15 execution path patterns:')
print(df['execution_path'].value_counts().head(15).to_string())

# Accuracy by path group
def path_group(ep_val):
    s = str(ep_val).lower()
    if s.startswith('biller'): return 'biller'
    if 'mcc' in s:             return 'mcc'
    if 'regex' in s:           return 'regex'
    if 'dt' in s:              return 'dt_only'
    return 'llm_only'

df['path_group'] = df['execution_path'].apply(path_group)
path_acc = df[l3_mask].groupby('path_group').agg(
    n=('l3_match', 'size'), l3_acc=('l3_match', 'mean')
).sort_values('l3_acc', ascending=False)
path_acc['l3_acc'] = path_acc['l3_acc'].map('{:.1%}'.format)
print('\nL3 accuracy by path group:')
display(path_acc)

## 8. Naive vs Pipeline Lift

In [ ]:
df_lift = df[l3_mask].dropna(subset=['gt_base_category_key', 'base_category_key', 'base_category_key_naive'])

pipeline_l3 = (df_lift['base_category_key']       == df_lift['gt_base_category_key']).mean()
naive_l3    = (df_lift['base_category_key_naive']  == df_lift['gt_base_category_key']).mean()
lift        = pipeline_l3 - naive_l3

print('Naive vs pipeline lift (L3, same evaluable rows):')
print(f'  Naive (no hints):  {naive_l3:.1%}')
print(f'  Full pipeline:     {pipeline_l3:.1%}')
print(f'  Lift:              +{lift:.1%}')

# Where does the pipeline specifically help vs hurt?
df_lift = df_lift.copy()
df_lift['naive_correct']    = df_lift['base_category_key_naive'] == df_lift['gt_base_category_key']
df_lift['pipeline_correct'] = df_lift['base_category_key']       == df_lift['gt_base_category_key']
df_lift['pipeline_helped']  =  df_lift['pipeline_correct'] & ~df_lift['naive_correct']
df_lift['pipeline_hurt']    = ~df_lift['pipeline_correct'] &  df_lift['naive_correct']

print(f'\nPipeline helped (naive wrong, pipeline right): {df_lift["pipeline_helped"].sum()}')
print(f'Pipeline hurt   (naive right, pipeline wrong):  {df_lift["pipeline_hurt"].sum()}')

if df_lift['pipeline_hurt'].sum() > 0:
    print('\nSample - pipeline hurt cases:')
    display(df_lift[df_lift['pipeline_hurt']][
        ['original_description', 'merchant', 'gt_base_category_key',
         'base_category_key_naive', 'base_category_key', 'execution_path', 'reason']
    ].head(15))

## 9. Merchant Extraction Coverage

In [ ]:
merchant_present = df['merchant'].notna() & (df['merchant'].str.strip() != '') & (df['merchant'].str.lower() != 'null')

print(f'Merchant extracted:     {merchant_present.sum()} ({merchant_present.mean()*100:.1f}%)')
print(f'No merchant extracted:  {(~merchant_present).sum()}')

# Accuracy when merchant present vs absent
n_present = (l3_mask & merchant_present).sum()
n_absent = (l3_mask & ~merchant_present).sum()
print(f'\nL3 accuracy when merchant present: {df[l3_mask & merchant_present]["l3_match"].mean():.1%}  (n={n_present})')
print(f'L3 accuracy when merchant absent:  {df[l3_mask & ~merchant_present]["l3_match"].mean():.1%}  (n={n_absent})')

print('\nTop 20 merchants extracted:')
print(df[merchant_present]['merchant'].value_counts().head(20).to_string())

## 10. Provider Breakdown

In [ ]:
prov = df[l3_mask].groupby('provider_name').agg(
    n        = ('l3_match', 'count'),
    l1_acc   = ('l1_match', 'mean'),
    l3_acc   = ('l3_match', 'mean'),
    avg_conf = ('confidence', 'mean'),
).sort_values('l3_acc', ascending=False).reset_index()

# L2a per provider (non-spend rows only)
prov_l2a = df[l3_mask & l2a_mask].groupby('provider_name').agg(
    n_ns     = ('l2a_match', 'count'),
    l2a_acc  = ('l2a_match', 'mean'),
).reset_index()

prov = prov.merge(prov_l2a, on='provider_name', how='left')

display(prov.assign(
    l1_acc   = prov['l1_acc'].map('{:.1%}'.format),
    l2a_acc  = prov['l2a_acc'].map(lambda x: f'{x:.1%}' if pd.notna(x) else 'n/a'),
    l3_acc   = prov['l3_acc'].map('{:.1%}'.format),
    avg_conf = prov['avg_conf'].map(lambda x: f'{x:.1f}' if pd.notna(x) else 'n/a'),
    n_ns     = prov['n_ns'].map(lambda x: f'{int(x)}' if pd.notna(x) else '0'),
)[['provider_name', 'n', 'l1_acc', 'l2a_acc', 'n_ns', 'l3_acc', 'avg_conf']])

low_provs = prov[prov['l3_acc'] < 0.5]['provider_name'].tolist()
if low_provs:
    print(f'\nProviders below 50% L3: {low_provs}')
    cols_review = ['original_description', 'merchant', 'gt_base_category_key',
                   'base_category_key', 'reason', 'execution_path']
    display(mismatches[mismatches['provider_name'].isin(low_provs)][cols_review].head(20))

## 11. Spend Type Mismatch Deep Dive

In [ ]:
l1_wrong = mismatches[~mismatches['l1_match'] & l1_mask.reindex(mismatches.index, fill_value=False)]
l1_right = mismatches[ mismatches['l1_match']]

if len(l1_wrong) > 0:
    print(f'=== L1 Wrong ({len(l1_wrong)} rows) ===')
    display(l1_wrong.groupby(['gt_spend_type_resolved', 'spend_type']).size().reset_index(name='count'))

    print('\nSample: GT=non_spend, predicted=spend')
    ns_wrong = l1_wrong[l1_wrong['gt_spend_type_resolved'] == 'non_spend']
    display(ns_wrong[['original_description', 'merchant', 'gt_base_category_key', 'spend_type', 'reason', 'execution_path']].head(10))

    print('\nSample: GT=spend, predicted=non_spend')
    s_wrong = l1_wrong[l1_wrong['gt_spend_type_resolved'] == 'spend']
    display(s_wrong[['original_description', 'merchant', 'gt_base_category_key', 'spend_type', 'reason', 'execution_path']].head(10))
else:
    print('No L1 wrong rows.')

## 12. Summary

In [ ]:
top_pair = (
    f"{pair_counts.iloc[0]['gt_base_category_key']} -> {pair_counts.iloc[0]['base_category_key']}"
    f" (n={pair_counts.iloc[0]['count']})"
) if len(pair_counts) else 'N/A'

print('=' * 55)
print('RUN 3 ANALYSIS SUMMARY')
print('=' * 55)
print(f'Total rows:              {len(df)}')
print(f'Parse errors:            {df["parse_error"].sum()} ({df["parse_error"].mean()*100:.1f}%)')
print(f'Invalid keys:            {df["invalid_key"].sum()} ({df["invalid_key"].mean()*100:.1f}%)')
print(f'Evaluable rows (L3):     {l3_mask.sum()}')
print(f'L1 spend_type:           {l1_acc:.1%}  ({l1_mask.sum()} evaluable)')
print(f'L2a category_type:       {l2a_acc:.1%}  ({l2a_mask.sum()} non-spend evaluable)')
print(f'L3 exact key:            {l3_acc:.1%}')
print(f'Naive L3:                {naive_l3:.1%}')
print(f'Pipeline lift:           +{lift:.1%}')
print(f'Avg confidence:          {df["confidence"].mean():.1f}')
print(f'Merchant extracted:      {merchant_present.sum()} ({merchant_present.mean()*100:.1f}%)')
print(f'Mismatches:              {len(mismatches)}')
print(f'  Wrong spend_type:      {len(l1_wrong)} ({len(l1_wrong)/max(len(mismatches),1):.0%})')
print(f'  Right type, wrong key: {len(l1_right)} ({len(l1_right)/max(len(mismatches),1):.0%})')
print(f'Top mismatch pair:       {top_pair}')
print('=' * 55)

## 13. Interactive Review Widget

In [ ]:
from ipywidgets import interact, IntSlider, ToggleButtons, Dropdown, HBox, VBox, Output

spend_opts = ['Any', 'spend', 'non_spend']
gt_opts    = ['Any'] + sorted(df['gt_base_category_key'].dropna().unique().tolist())
pred_opts  = ['Any'] + sorted(df['base_category_key'].dropna().unique().tolist())
path_opts  = ['Any'] + sorted(df['path_group'].dropna().unique().tolist())

subset_toggle = ToggleButtons(options=['All rows', 'Mismatches only'], value='All rows')
spend_dd      = Dropdown(options=spend_opts, value='Any', description='Spend type:')
gt_dd         = Dropdown(options=gt_opts,    value='Any', description='GT key:')
pred_dd       = Dropdown(options=pred_opts,  value='Any', description='Predicted:')
path_dd       = Dropdown(options=path_opts,  value='Any', description='Path group:')
idx_slider    = IntSlider(min=0, max=len(df)-1, step=1, value=0, description='Row:')
out           = Output()

def get_filtered():
    data = mismatches if subset_toggle.value == 'Mismatches only' else df
    if spend_dd.value  != 'Any': data = data[data['spend_type']            == spend_dd.value]
    if gt_dd.value     != 'Any': data = data[data['gt_base_category_key']  == gt_dd.value]
    if pred_dd.value   != 'Any': data = data[data['base_category_key']     == pred_dd.value]
    if path_dd.value   != 'Any': data = data[data['path_group']            == path_dd.value]
    return data.reset_index(drop=True)

def update_slider(*args):
    data = get_filtered()
    idx_slider.max = max(len(data) - 1, 0)
    idx_slider.value = 0
    render()

def render(*args):
    data = get_filtered()
    out.clear_output(wait=True)
    with out:
        if len(data) == 0:
            print('No rows match filters.')
            return
        idx = min(idx_slider.value, len(data) - 1)
        row = data.iloc[idx]
        status = 'MATCH' if row.get('l3_match') else 'MISMATCH'
        print(f'Showing {idx + 1} of {len(data)} rows')
        print('=' * 62)
        print(f'Row {idx} | {status} | Confidence: {row.get("confidence")}')
        print('=' * 62)
        print(f'Provider:      {row.get("provider_name")}')
        print(f'Description:   {row.get("original_description")}')
        print(f'Merchant:      {row.get("merchant")}')
        print(f'Amount:        {row.get("amount")}')
        print()
        print(f'-- Pipeline ------------------------------------------')
        print(f'Execution path: {row.get("execution_path")}')
        print(f'Reason:         {row.get("reason")}')
        print(f'Flag:           {row.get("flag")}')
        print()
        print(f'-- LLM Output ----------------------------------------')
        print(f'spend_type:     {row.get("spend_type")}')
        print(f'category_group: {row.get("category_group")}')
        print(f'category_type:  {row.get("category_type")}')
        print(f'Predicted:      {row.get("base_category_key")}')
        print(f'Naive:          {row.get("base_category_key_naive")}')
        print(f'Ground truth:   {row.get("gt_base_category_key")}')

for w in [subset_toggle, spend_dd, gt_dd, pred_dd, path_dd]:
    w.observe(update_slider, names='value')
idx_slider.observe(render, names='value')

display(VBox([subset_toggle, HBox([spend_dd, gt_dd, pred_dd, path_dd]), idx_slider, out]))
update_slider()

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

# ── Build pivot ───────────────────────────────────────────────────────────────
mismatch_df = df[~df['l3_match'] & l3_mask].copy()

top_gt   = mismatch_df['gt_base_category_key'].value_counts().head(20).index.tolist()
top_pred = mismatch_df['base_category_key'].value_counts().head(20).index.tolist()

top_pairs = (
    mismatch_df.groupby(['gt_base_category_key', 'base_category_key'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

pivot = top_pairs.pivot_table(
    index='gt_base_category_key', columns='base_category_key', values='count', fill_value=0
)

pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

gt_totals  = df[l3_mask]['gt_base_category_key'].value_counts()
row_labels = [
    f"{cat} (n={gt_totals.get(cat, 0)})"
    for cat in pivot.index
]
col_labels = list(pivot.columns)

# ── Labels store ──────────────────────────────────────────────────────────────
labels_store = {}   # (gt, pred, row_idx) -> {verdict, notes}
combo_store  = {}   # (gt, pred)          -> {verdict, notes}

# ── Heatmap figure ────────────────────────────────────────────────────────────
def make_heatmap(use_pct):
    data      = pivot_pct.values if use_pct else pivot.values
    text_vals = [[f"{v:.0f}%" if use_pct else str(int(v)) if v > 0 else ""
                  for v in row] for row in data]
    return go.Heatmap(
        z=data,
        x=col_labels,
        y=row_labels,
        colorscale="YlOrRd",
        text=text_vals,
        texttemplate="%{text}",
        hovertemplate="GT: %{y}<br>Pred: %{x}<br>" +
                      ("%{z:.1f}%" if use_pct else "%{z:.0f}") + "<extra></extra>",
        showscale=True,
        colorbar=dict(title="% of GT category" if use_pct else "Count"),
    )

fig = go.FigureWidget(make_heatmap(use_pct=True))
fig.update_layout(
    title="Click a cell to review transactions",
    xaxis=dict(title="Predicted key", tickangle=90, tickfont=dict(size=9)),
    yaxis=dict(title="Ground truth",  tickfont=dict(size=9)),
    height=800,
    width=1000,
    margin=dict(l=280, b=220, t=50),
)

# ── Toggle ────────────────────────────────────────────────────────────────────
toggle = widgets.ToggleButtons(
    options=["% of GT category", "Count"],
    description="Show:",
    layout=widgets.Layout(margin="8px 0"),
)

def on_toggle(change):
    use_pct = change["new"] == "% of GT category"
    with fig.batch_update():
        new_trace = make_heatmap(use_pct)
        fig.data[0].z             = new_trace.z
        fig.data[0].text          = new_trace.text
        fig.data[0].texttemplate  = new_trace.texttemplate
        fig.data[0].hovertemplate = new_trace.hovertemplate
        fig.data[0].colorbar      = new_trace.colorbar

toggle.observe(on_toggle, names="value")

# ── Review panel ──────────────────────────────────────────────────────────────
selection_label = widgets.HTML("<b>Click a cell in the heatmap to load transactions</b>")
progress_html   = widgets.HTML("")

# ── Combo-level verdict ───────────────────────────────────────────────────────
combo_header  = widgets.HTML("<b>Combo-level label</b> (applies to all transactions in this cell)")
combo_verdict = widgets.Dropdown(
    options=["", "GT_ERROR", "LLM_ERROR", "TAXONOMY_GAP", "AMBIGUOUS", "MIXED"],
    description="Verdict:",
    layout=widgets.Layout(width="300px"),
)
combo_notes       = widgets.Textarea(
    placeholder="Notes about this GT -> prediction pair overall...",
    description="Notes:",
    layout=widgets.Layout(width="95%", height="55px"),
)
combo_save_btn    = widgets.Button(description="Save combo label", button_style="warning", icon="tag")
combo_save_status = widgets.HTML("")

# ── Transaction-level verdict ─────────────────────────────────────────────────
txn_header   = widgets.HTML("<b>Transaction-level label</b>")
txn_selector = widgets.Dropdown(description="Transaction:", layout=widgets.Layout(width="95%"))
txn_detail   = widgets.HTML("")
txn_verdict  = widgets.Dropdown(
    options=["", "LLM_CORRECT", "GT_CORRECT", "BOTH_OK", "NEITHER"],
    description="Verdict:",
    layout=widgets.Layout(width="300px"),
)
txn_notes       = widgets.Textarea(
    placeholder="Optional notes for this transaction...",
    description="Notes:",
    layout=widgets.Layout(width="95%", height="55px"),
)
txn_save_btn    = widgets.Button(description="Save txn label", button_style="success", icon="check")
txn_save_status = widgets.HTML("")

# ── Export ────────────────────────────────────────────────────────────────────
export_btn    = widgets.Button(description="Export all labels", button_style="primary", icon="download")
export_status = widgets.HTML("")

# ── State ─────────────────────────────────────────────────────────────────────
current_gt   = [None]
current_pred = [None]
current_txns = [None]

# ── Helpers ───────────────────────────────────────────────────────────────────
def fmt_detail(row):
    fields = {
        "Provider":       row.get("provider_name", ""),
        "Amount":         row.get("amount", ""),
        "Description":    row.get("original_description", ""),
        "Merchant":       row.get("merchant", ""),
        "Execution path": row.get("execution_path", ""),
        "Confidence":     row.get("confidence", ""),
        "Reason":         row.get("reason", ""),
        "Naive pred":     row.get("base_category_key_naive", ""),
        "Ground truth":   f"<b style='color:#e74c3c'>{row.get('gt_base_category_key','')}</b>",
        "LLM prediction": f"<b style='color:#2980b9'>{row.get('base_category_key','')}</b>",
        "Flag":           row.get("flag", ""),
    }
    rows = "".join(
        f"<tr><td style='padding:3px 10px;color:#666;white-space:nowrap'><b>{k}</b></td>"
        f"<td style='padding:3px'>{v}</td></tr>"
        for k, v in fields.items()
    )
    return f"<table style='font-size:13px;border-collapse:collapse'>{rows}</table>"

def update_progress():
    gt, pred = current_gt[0], current_pred[0]
    txns = current_txns[0]
    if txns is None or len(txns) == 0:
        return
    labelled   = sum(1 for idx in txns["row_idx"] if (gt, pred, idx) in labels_store)
    combo_done = "🏷 combo labelled" if (gt, pred) in combo_store else ""
    progress_html.value = (
        f"Txn labels: <b>{labelled} / {len(txns)}</b> &nbsp;&nbsp; {combo_done}"
    )

def refresh_txn_dropdown():
    gt, pred = current_gt[0], current_pred[0]
    txns = current_txns[0]
    if txns is None:
        return
    cur = txn_selector.value
    txn_selector.options = [
        (f"[{'✅' if (gt, pred, row['row_idx']) in labels_store else '  '}]  "
         f"row {row['row_idx']}  |  {str(row.get('original_description',''))[:60]}",
         i)
        for i, (_, row) in enumerate(txns.iterrows())
    ]
    if cur is not None and cur < len(txns):
        txn_selector.value = cur

def load_cell(gt_label, pred_label):
    gt_clean   = gt_label.split(" (n=")[0]
    pred_clean = pred_label
    current_gt[0]   = gt_clean
    current_pred[0] = pred_clean

    txns = df[
        (df["gt_base_category_key"] == gt_clean) &
        (df["base_category_key"]    == pred_clean)
    ].reset_index(drop=True)
    current_txns[0] = txns

    selection_label.value = (
        f"<b style='font-size:14px'>"
        f"GT: <span style='color:#e74c3c'>{gt_clean}</span> &nbsp;->;&nbsp; "
        f"Pred: <span style='color:#2980b9'>{pred_clean}</span>"
        f" &nbsp; ({len(txns)} transactions)</b>"
    )

    existing_combo      = combo_store.get((gt_clean, pred_clean), {})
    combo_verdict.value = existing_combo.get("verdict", "")
    combo_notes.value   = existing_combo.get("notes",   "")
    combo_save_status.value = ""

    if len(txns) == 0:
        txn_selector.options = []
        txn_detail.value     = ""
        update_progress()
        return

    txn_selector.options = [
        (f"[{'✅' if (gt_clean, pred_clean, row['row_idx']) in labels_store else '  '}]  "
         f"row {row['row_idx']}  |  {str(row.get('original_description',''))[:60]}",
         i)
        for i, (_, row) in enumerate(txns.iterrows())
    ]
    txn_selector.value = 0
    update_progress()

# ── Event handlers ────────────────────────────────────────────────────────────
def on_click(trace, points, state):
    if not points.point_inds:
        return
    load_cell(points.ys[0], points.xs[0])

fig.data[0].on_click(on_click)

def on_txn_select(change):
    idx = change["new"]
    if idx is None or current_txns[0] is None:
        return
    row = current_txns[0].iloc[idx]
    txn_detail.value = fmt_detail(row)
    key = (current_gt[0], current_pred[0], row["row_idx"])
    existing          = labels_store.get(key, {})
    txn_verdict.value = existing.get("verdict", "")
    txn_notes.value   = existing.get("notes",   "")
    txn_save_status.value = ""

txn_selector.observe(on_txn_select, names="value")

def on_save_combo(b):
    gt, pred = current_gt[0], current_pred[0]
    if gt is None:
        return
    combo_store[(gt, pred)] = {"verdict": combo_verdict.value, "notes": combo_notes.value}
    combo_save_status.value = "✅ Saved"
    update_progress()

combo_save_btn.on_click(on_save_combo)

def on_save_txn(b):
    idx = txn_selector.value
    if idx is None or current_txns[0] is None:
        return
    row = current_txns[0].iloc[idx]
    key = (current_gt[0], current_pred[0], row["row_idx"])
    labels_store[key]     = {"verdict": txn_verdict.value, "notes": txn_notes.value}
    txn_save_status.value = "✅ Saved"
    refresh_txn_dropdown()
    update_progress()

txn_save_btn.on_click(on_save_txn)

def on_export(b):
    combo_rows = [
        {"type": "combo", "gt_base_category_key": gt, "base_category_key": pred,
         "row_idx": "", "verdict": v["verdict"], "notes": v["notes"]}
        for (gt, pred), v in combo_store.items()
    ]
    txn_rows = [
        {"type": "transaction", "gt_base_category_key": gt, "base_category_key": pred,
         "row_idx": idx, "verdict": v["verdict"], "notes": v["notes"]}
        for (gt, pred, idx), v in labels_store.items()
    ]
    out = pd.DataFrame(combo_rows + txn_rows)
    if len(out) == 0:
        export_status.value = "No labels saved yet"
        return
    out.to_csv(OUTPUT_DIR / "run3_manual_review_labels.csv", index=False)
    export_status.value = (
        f"✅ Exported {len(combo_rows)} combo + {len(txn_rows)} txn labels "
        f"-> outputs/run3_manual_review_labels.csv"
    )

export_btn.on_click(on_export)

# ── Layout ────────────────────────────────────────────────────────────────────
combo_panel = widgets.VBox([
    combo_header,
    widgets.HBox([combo_verdict, combo_save_btn, combo_save_status]),
    combo_notes,
], layout=widgets.Layout(padding="8px", border="1px solid #f39c12",
                         border_radius="4px", margin="6px 0"))

txn_panel = widgets.VBox([
    txn_header,
    txn_selector,
    txn_detail,
    widgets.HBox([txn_verdict, txn_save_btn, txn_save_status]),
    txn_notes,
], layout=widgets.Layout(padding="8px", border="1px solid #27ae60",
                         border_radius="4px", margin="6px 0"))

review_panel = widgets.VBox([
    selection_label,
    progress_html,
    combo_panel,
    txn_panel,
    widgets.HBox([export_btn, export_status]),
], layout=widgets.Layout(padding="10px", width="100%"))

display(widgets.VBox([toggle, fig, review_panel]))

In [ ]:

# ── Spend accuracy breakdown ──────────────────────────────────────────────────
OUTPUT_DIR = "outputs"
df_eval = df[l3_mask].copy()
total_eval = len(df_eval)

n_exact    = df_eval['l3_match'].sum()
n_sp_ok    = (~df_eval['l3_match'] &  df_eval['l1_match'] & l1_mask.reindex(df_eval.index, fill_value=False)).sum()
n_sp_wrong = (~df_eval['l3_match'] & ~df_eval['l1_match'] & l1_mask.reindex(df_eval.index, fill_value=False)).sum()
n_inv_keys = df['invalid_key'].sum()   # raw count from full df, not l3_mask subset
n_unkn     = (~l1_mask & l3_mask).sum()

print(f"L1 spend_type accuracy (all evaluable):    {df_eval.loc[l1_mask.reindex(df_eval.index, fill_value=False), 'l1_match'].mean()*100:.1f}%")
print(f"L1 spend_type accuracy (mismatches only):  {df_eval.loc[~df_eval['l3_match'] & l1_mask.reindex(df_eval.index, fill_value=False), 'l1_match'].mean()*100:.1f}%")
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: stacked breakdown of all evaluable rows ─────────────────────────────
labels = ["base_category_key\ncorrect", "Wrong base_category_key,\ncorrect spend_type", "Wrong base_category_key,\nwrong spend_type",
          "Invalid\nbase_category_key", "Unknown\nspend_type"]
sizes  = [int(n_exact), int(n_sp_ok), int(n_sp_wrong), int(n_inv_keys), int(n_unkn)]
colors = ["#2ecc71", "#a8d8a8", "#e74c3c", "#c0392b", "#bdc3c7"]
axes[0].bar(labels, sizes, color=colors)
for i, s in enumerate(sizes):
    axes[0].text(i, s + 5, f"{s}\n({s/total_eval*100:.1f}%)", ha="center", fontsize=8)
axes[0].set_title(f"base_category_key prediction breakdown - {total_eval:,} evaluable rows", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Row count")
axes[0].set_ylim(0, max(sizes) * 1.25)

# ── Right: spend acc vs exact acc for worst 15 categories ────────────────────
l1_mask_eval = l1_mask.reindex(df_eval.index, fill_value=False)
sp_by_cat = (
    df_eval[l1_mask_eval]
    .groupby('gt_base_category_key')
    .agg(
        n        = ('l1_match', 'count'),
        spend_ok = ('l1_match', 'sum'),
        exact_ok = ('l3_match', 'sum'),
    )
    .query("n >= 5")
    .assign(
        spend_acc = lambda x: x['spend_ok'] / x['n'] * 100,
        exact_acc = lambda x: x['exact_ok'] / x['n'] * 100,
    )
    .sort_values('spend_acc')
    .head(15)
    .reset_index()
)
y = range(len(sp_by_cat))
axes[1].barh(y,                     sp_by_cat['spend_acc'], color="#3498db", height=0.4, label="L1 spend_type acc")
axes[1].barh([i + 0.4 for i in y],  sp_by_cat['exact_acc'], color="#e67e22", height=0.4, label="L3 exact key acc")
axes[1].set_yticks([i + 0.2 for i in y])
axes[1].set_yticklabels(sp_by_cat['gt_base_category_key'], fontsize=8)
axes[1].set_xlabel("Accuracy (%)")
axes[1].set_title("L1 spend_type acc vs L3 base_category_key acc\n(worst 15 categories by L1)", fontsize=11, fontweight="bold")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("outputs/run3_spend_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:

prov_detail = (
    df[l3_mask].groupby('provider_name')
    .agg(
        total    = ('l3_match', 'count'),
        correct  = ('l3_match', 'sum'),
        invalid  = ('invalid_key', 'sum'),
    )
    .assign(
        accuracy    = lambda x: (x['correct']  / x['total'] * 100).round(1),
        invalid_pct = lambda x: (x['invalid']  / x['total'] * 100).round(1),
        wrong_valid = lambda x: x['total'] - x['correct'] - x['invalid'],
    )
    .sort_values('accuracy')
    .reset_index()
)

n_per_provider = df[l3_mask]['provider_name'].value_counts().max()

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(prov_detail))
ax.bar(x, prov_detail['correct'],     label="Correct",           color="#2ecc71")
ax.bar(x, prov_detail['wrong_valid'], label="Wrong (valid key)", color="#e67e22",
       bottom=prov_detail['correct'])
ax.bar(x, prov_detail['invalid'],     label="Invalid key",       color="#e74c3c",
       bottom=prov_detail['correct'] + prov_detail['wrong_valid'])
ax.set_xticks(x)
ax.set_xticklabels(prov_detail['provider_name'], rotation=35, ha="right", fontsize=9)
ax.axhline(n_per_provider, color="black", linewidth=0.8, linestyle="--",
           label=f"max n={n_per_provider} per provider")
ax.set_ylabel("Row count")
ax.set_title("Per-provider base_category_key prediction breakdown (sorted by L3 accuracy)", fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("outputs/run3_provider_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

print(prov_detail[['provider_name', 'total', 'correct', 'accuracy',
                    'wrong_valid', 'invalid', 'invalid_pct']].to_string(index=False))